# SUSHRUTA — Minimal GPU Verification

## Executive Summary
This notebook performs a minimal GPU verification for **SUSHRUTA Baseline-01** prior to full model training on GPU.

### Verification Checklist:
- **PyTorch & CUDA Diagnostics**: Confirm PyTorch version (`2.8.0+cu128`), CUDA runtime (`12.8`), and GPU model (`NVIDIA GeForce RTX 2050`).
- **Basic Tensor CUDA Execution**: Create a CPU tensor, transfer to GPU (`cuda:0`), and execute matrix multiplication on CUDA.
- **U-Net Architecture CUDA Execution**: Instantiate `UNet2D`, transfer model weights to CUDA, run inference pass on a `[1, 1, 256, 256]` dummy tensor.
- **Shape & Memory Integrity**: Verify output shape `[1, 1, 256, 256]` and measure allocated VRAM.
- **Non-destructive Verification**: No training executed; dataset left untouched.

---
## Section 1: Environment Setup & CUDA Diagnostics
Checking PyTorch installation, CUDA availability, CUDA runtime version, GPU device details, and count.

In [1]:
import os
import sys
import torch

# Add project root to sys.path
current_dir = os.path.abspath('')
project_root = current_dir if os.path.exists(os.path.join(current_dir, 'src')) else os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA runtime version: {torch.version.cuda}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"GPU device count: {torch.cuda.device_count()}")


PyTorch version: 2.8.0+cu128
CUDA available: True
CUDA runtime version: 12.8
GPU name: NVIDIA GeForce RTX 2050
GPU device count: 1


---
## Section 2: Tensor Creation & CUDA Execution
Creating a CPU tensor, moving it to CUDA, performing matrix multiplication on GPU, and verifying tensor device `cuda:0`.

In [2]:
# Create small random tensor on CPU
tensor_cpu = torch.randn(100, 100)
print(f"CPU tensor device: {tensor_cpu.device}")

# Move tensor to CUDA
tensor_gpu = tensor_cpu.to('cuda')
print(f"CUDA tensor device: {tensor_gpu.device}")

# Perform matrix multiplication on GPU
result_gpu = torch.matmul(tensor_gpu, tensor_gpu)

# Verify resulting tensor's device is cuda:0
print(f"Result tensor device: {result_gpu.device}")
assert 'cuda' in str(result_gpu.device), f"Expected CUDA device, got {result_gpu.device}"
print("Tensor CUDA execution: PASSED")


CPU tensor device: cpu
CUDA tensor device: cuda:0
Result tensor device: cuda:0
Tensor CUDA execution: PASSED


---
## Section 3: U-Net Architecture GPU Verification
Instantiating current SUSHRUTA U-Net (`UNet2D`), moving model to CUDA, creating `[1, 1, 256, 256]` dummy input on CUDA, and executing one inference forward pass.

In [3]:
from src.models.unet import UNet2D

# Instantiate current SUSHRUTA U-Net architecture
model = UNet2D(in_channels=1, out_channels=1, base_channels=32)

# Move model to CUDA
model = model.to('cuda')

# Create dummy input with shape [1, 1, 256, 256]
dummy_input = torch.randn(1, 1, 256, 256, device='cuda')

# Run one forward pass under inference mode
model.eval()
with torch.inference_mode():
    output = model(dummy_input)

# Inspect device, output shape, and GPU memory allocated
input_device = dummy_input.device
model_device = next(model.parameters()).device
output_shape = list(output.shape)
gpu_memory_mb = torch.cuda.memory_allocated() / (1024 ** 2)

print(f"Input device: {input_device}")
print(f"Model device: {model_device}")
print(f"Output shape: {output_shape}")
print(f"GPU memory allocated: {gpu_memory_mb:.2f} MB")

# Verify output shape is [1, 1, 256, 256]
assert output_shape == [1, 1, 256, 256], f"Expected shape [1, 1, 256, 256], got {output_shape}"
print("U-Net CUDA execution: PASSED")


Input device: cuda:0
Model device: cuda:0
Output shape: [1, 1, 256, 256]
GPU memory allocated: 25.21 MB
U-Net CUDA execution: PASSED


---
## Section 4: Final Verification Summary
Prints official verification report confirming system readiness for GPU training.

In [4]:
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unknown GPU"

print("========================================")
print("SUSHRUTA GPU VERIFICATION")
print("========================================")
print("CUDA: PASSED")
print(f"GPU: {gpu_name}")
print("Tensor CUDA execution: PASSED")
print("U-Net CUDA execution: PASSED")
print(f"Output shape: {output_shape}")
print("========================================")
print("READY FOR GPU TRAINING")
print("========================================")


SUSHRUTA GPU VERIFICATION
CUDA: PASSED
GPU: NVIDIA GeForce RTX 2050
Tensor CUDA execution: PASSED
U-Net CUDA execution: PASSED
Output shape: [1, 1, 256, 256]
READY FOR GPU TRAINING
